# Combining Data Wrap-up
v.ekc

Standard merges assume clean, matching timestamps — but real-world data rarely cooperates. Today we cover specialized pandas tools for joining time series at different frequencies and approximating temporal matches.

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Combining Time Series Data |
| 3 | Approximate Temporal Matches — merge_asof |
| 4 | Spatial + Temporal Matching |
| 5 | 🔬 Activity — Bird GPS Data |
| Appendix | Quick Reference |

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd

---
## 2. Combining Time Series Data

When time series data comes from different sensors or at different frequencies, a plain `.merge()` may silently sort incorrectly or leave gaps. `merge_ordered` and `resample` give you more control.

### 📋 Board Reference

| Function | What it does |
|----------|--------------|
| `pd.merge_ordered(df1, df2)` | Merge and sort by the key column |
| `pd.merge_ordered(..., fill_method='ffill')` | Forward-fill gaps after merging |
| `df.resample('1H', on='time').sum()` | Aggregate to a coarser time resolution |
| `df.groupby(pd.Grouper(key='time', freq='1H')).sum()` | Group by time — safe when other cols shouldn't be summed |

In [ ]:
env_data = pd.DataFrame({
    'time': pd.date_range('2025-04-14 08:00', periods=6, freq='1h'),
    'temperature_C': [14.0, 12.5, 13.0, 13.5, 12.0, 14.2],
    'humidity_pct': [85, 83, 80, 78, 76, 75]
})
env_data.sort_values(by='temperature_C',inplace=True)
env_data

In [ ]:
precip_data = pd.DataFrame({
    'time': pd.date_range('2025-04-14 07:00', periods=9, freq='1h'),
    'precipitation': [0.1, 0.3, 0, 0, 0.1, 0.05, 0.05, 0.1, 0.1],
})
precip_data

In [ ]:
# A standard merge goes in the order of the original left dataframe
pd.merge(env_data, precip_data)

In [ ]:
# merge_ordered sorts the data too
pd.merge_ordered(env_data, precip_data)

In [ ]:
# merge_ordered also allows you to fill in gaps with a method of choice
pd.merge_ordered(env_data, precip_data, fill_method='ffill')

In [ ]:
# When this is most useful?
precip_data = pd.DataFrame({
    'time': pd.date_range('2025-04-14 07:00', periods=10, freq='30min'),
    'precipitation': [0.1, 0.3, 0, 0, 0.1, 0.05, 0.05, 0.1, 0.1, 0.2],
})
precip_data

In [ ]:
# When the frequency of data points is different
env_data

In [ ]:
# When the frequency of data points is different
pd.merge_ordered(env_data, precip_data)

In [ ]:
pd.merge_ordered(env_data, precip_data, fill_method='ffill')

In [ ]:
# Merge_ordered also accepts different "how" args
pd.merge_ordered(env_data, precip_data, how = 'inner')

In [ ]:
# You can also take the approach of "course graining" with resample
hourly_precip = precip_data.resample('1H', on='time').sum()
hourly_precip

In [ ]:
# Merge at course-grained resolution
course = pd.merge_ordered(env_data,hourly_precip.reset_index())
course

In [ ]:
# When should we be careful about resampling?
precip_data['lat'] = [38.1, 38.1, 38.1, 38.1, 38.1, 38.2, 38.2, 38.2, 38.2, 38.2]
precip_data['lon'] = [-120.1, -120.1, -120.2,-120.2, -120.2,-120.1, -120.1, -120.2,-120.2, -120.2]
precip_data

In [ ]:
# Resampling this would lead to inaccurate location data
precip_data.resample('1H',on='time').sum()

In [ ]:
# In this case, use groupby with pd.Grouper
precip_data.groupby(pd.Grouper(key = 'time', freq="1H")).sum()

In [ ]:
# Groupby can handle multiple columns
precip_data.groupby([pd.Grouper(key = 'time', freq="1H"),'lat','lon']).sum()

In [ ]:
# Get course-grained data for each location
precip_data.groupby([pd.Grouper(key = 'time', freq="1H"),'lat','lon']).mean().reset_index()

---
## 3. Approximate Temporal Matches — merge_asof

What if your two datasets have timestamps that never align exactly? `merge_asof` matches each left timestamp to the nearest (or most recent) right timestamp.

### 📋 Board Reference

| What | Code |
|------|------|
| Match to most recent prior event | `pd.merge_asof(left, right, on='time')` |
| Match to nearest event | `pd.merge_asof(..., direction='nearest')` |
| Match to next future event | `pd.merge_asof(..., direction='forward')` |
| Different column names | `pd.merge_asof(..., left_on='t1', right_on='t2')` |
| Require close match | `pd.merge_asof(..., tolerance=pd.Timedelta('1h'))` |

In [ ]:
# Irregular observational data
observations = pd.DataFrame({
    'time': pd.to_datetime([
        '2025-04-14 08:03', '2025-04-14 09:21', '2025-04-14 11:00',
        '2025-04-14 09:45', '2025-04-14 13:27'
    ]),
    'animal': ['fox', 'bear', 'wolf', 'bear', 'wolf'],
    'location': ['ridge', 'valley', 'creek', 'creek', 'ridge']
}).sort_values('time')

observations

In [ ]:
env_data = pd.DataFrame({
    'time': pd.date_range('2025-04-14 08:00', periods=6, freq='1h'),
    'temperature_C': [12.0, 12.5, 13.0, 13.5, 14.0, 14.2],
    'humidity_pct': [85, 83, 80, 78, 76, 75]
})
env_data

In [ ]:
# Merge ordered takes all unique
pd.merge_ordered(observations, env_data)

In [ ]:
# Merge ordered incorrectly duplicates observations
pd.merge_ordered(observations, env_data, fill_method='ffill')

In [ ]:
# Remind ourselves of what the originals look like
observations

In [ ]:
env_data

In [ ]:
# Merge the left time points to the previous point from the right
pd.merge_asof(observations, env_data, on='time')

In [ ]:
# Merge the left time points to the nearest point from the right 
pd.merge_asof(observations, env_data, on='time', direction='nearest')

In [ ]:
# Merge the left time points to the next point from the right
pd.merge_asof(observations, env_data, on='time', direction='forward')

In [ ]:
# If they have different names
env_data.rename(columns={'time':'Time'},inplace=True)
pd.merge_asof(observations, env_data, left_on='time', right_on = 'Time')

In [ ]:
# Reset
env_data.rename(columns={'Time':'time'},inplace=True)

---
## 4. Spatial + Temporal Matching

When data has both location and time, you can constrain `merge_asof` to match within the same spatial group — and even set a time tolerance.

In [ ]:
observations['lat'] = [38.2, 38.2, 38.2, 38.2, 38.2]
observations['lon'] = [-120.1, -120.1, -120.2,-120.2, -120.2]

In [ ]:
env_data['lat'] = [38.2, 38.2, 38.2, 38.2, 38.2,38.2]
env_data['lon'] = [-120.1, -120.1, -120.2,-120.2, -120.2, -120.1]

In [ ]:
observations

In [ ]:
env_data

In [ ]:
# Merge each location on the time column
pd.merge_asof(observations, env_data, on='time')

In [ ]:
pd.merge_asof(observations, env_data, on='time', by = ['lat','lon'])

This could change the distance it reaches to get a match.

In [ ]:
# Set a tolerance for how close the points have to be to match
pd.merge_asof(observations, env_data, on='time', by = ['lat','lon'],
              tolerance=pd.Timedelta('1h'))

---
## 5. 🔬 Activity — Bird GPS Data

You're analyzing bird behavior with GPS trackers and sound-activated recorders. The GPS trackers log positions every 5 minutes. The sound recorders log audio events (calls, songs, other birds) at irregular times.

For each bird location, merge the most recent sound event recorded in that area.

---
### Try It 1: Merge Bird GPS + Sound Events 😊

1. **Inspect** `bird_gps` and `sound_events` — what are the timestamps and how do they align?

2. **Use `merge_asof`** to attach the most recent sound event to each GPS position.

3. **Try `direction='nearest'`** — does the result change? Which direction makes more biological sense for this use case?

In [ ]:
bird_gps = pd.DataFrame({
    "time": pd.to_datetime([
        "2023-04-13 08:00", "2023-04-13 08:05", "2023-04-13 08:10"
    ]),
    "bird_id": [101, 101, 101],
    "lat": [40.5, 40.5, 40.5],
    "lon": [-123.8, -123.8, -123.8]
})

sound_events = pd.DataFrame({
    "time": pd.to_datetime([
        "2023-04-13 07:59", "2023-04-13 08:03", "2023-04-13 08:12"
    ]),
    "event_type": ["call", "song", "flight"]
})

In [ ]:
# 1. Inspect the two datasets


In [ ]:
# 2. merge_asof: most recent sound event per GPS point


In [ ]:
# 3. Try direction='nearest'


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`direction='backward'` (default) makes more biological sense — we attach the sound that was recorded before the bird's position was logged, not after.*

```python
# 2. Most recent prior sound event
pd.merge_asof(bird_gps, sound_events, on='time')

# 3. Nearest sound event
pd.merge_asof(bird_gps, sound_events, on='time', direction='nearest')
# The 08:10 GPS point now gets 'flight' (at 08:12) instead of 'song' (at 08:03)
```

</details>

---
## Appendix — Time Series Combining Quick Reference

### merge_ordered
```python
pd.merge_ordered(df1, df2)                          # merge and sort by key
pd.merge_ordered(df1, df2, fill_method='ffill')     # forward-fill gaps
pd.merge_ordered(df1, df2, how='inner')             # inner join, sorted
```

### Resampling
```python
df.resample('1H', on='time').sum()                  # coarsen to hourly totals
# Use groupby when other columns shouldn't be aggregated:
df.groupby(pd.Grouper(key='time', freq='1H')).sum()
df.groupby([pd.Grouper(key='time', freq='1H'), 'lat', 'lon']).mean()
```

### merge_asof
```python
pd.merge_asof(left, right, on='time')               # match to most recent prior
pd.merge_asof(left, right, on='time',
              direction='nearest')                  # match to nearest
pd.merge_asof(left, right, on='time',
              direction='forward')                  # match to next future
pd.merge_asof(left, right, on='time',
              by=['lat','lon'])                      # match within groups
pd.merge_asof(left, right, on='time',
              tolerance=pd.Timedelta('1h'))          # require close match
```